# YOLOv11 - Yer seviyesi - catlak / yapisal hasar tespiti

**Katman:** Zorunlu

Guncel surum - v8 -> v11 kazanimini olcer.

Bizim yontem: 2x YOLOv8 (catlak.pt + bina.pt) + hibrit kip, guven esigi 0,20, gecikme 153-174 ms.  
Butun modeller `common.py` uzerinden AYNI Roboflow seti / AYNI bolme / AYNI metrik.  
Metrik: mAP@50, mAP@50-95, Kesinlik, Duyarlilik, F1, sinif bazli AP + FPS/Params/GFLOPs.

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/03_yer_seviyesi
    %pip install -q ultralytics roboflow
print('cwd:', os.getcwd())

## 1. Veri (Roboflow YOLO export)

In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd()))
import common as C
C.CONFIG.update(dict(
    roboflow_api_key='BURAYA_ROBOFLOW_API_KEY',
    roboflow_workspace='workspace-slug',
    roboflow_project='project-slug',
    roboflow_version=1,
))
ds_dir = C.download_dataset('yolov8')
DATA_YAML = ds_dir / 'data.yaml'
print('veri:', DATA_YAML)

## 2. Egit + degerlendir + kaydet

In [ ]:
MODEL_NAME = "yolo11s"
WEIGHTS = "yolo11s.pt"
TASK = "detect"
metrics, eff = C.train_yolo(WEIGHTS, DATA_YAML, MODEL_NAME, C.CONFIG, task=TASK)
print('mAP50:', round(metrics['map50'], 4), ' mAP50-95:', round(metrics['map50_95'], 4))
print('F1:', round(metrics['f1'], 4), ' FPS:', round(eff['fps'], 1))

## 3. Liderlik tablosu

In [ ]:
import pandas as pd; pd.read_csv('results/leaderboard.csv')